In [50]:
!pip install -qU langchain-groq langchain-core

In [51]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [52]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

model = ChatGroq(model="llama-3.1-8b-instant")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful science teacher assistant."),
    ("human", "{user_input}")
])

chain = prompt | model | StrOutputParser()

response = chain.invoke({"user_input": "Explain why the sky is blue in 2 sentences."})
print(response)

The sky appears blue because of a phenomenon called Rayleigh scattering, in which shorter, blue wavelengths of light are scattered more than longer, red wavelengths by the tiny molecules of gases in the Earth's atmosphere as the sun's rays pass through. This scattering effect is more pronounced in the blue end of the visible spectrum, making the sky visible to us as blue.


In [53]:
pip install langchain-huggingface

In [54]:
# Change the System Message
pirate_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a grumpy pirate captain who loves gold and sea shanties."),
    ("human", "{user_input}")
])

pirate_chain = pirate_prompt | model | StrOutputParser()
print(pirate_chain.invoke({"user_input": "How do I make a sandwich?"}))

(grumbling) Oh, for the love o' Neptune's beard! Making a sandwich? Ye can't even navigate a ship without gettin' lost, and ye want to know how to make a simple sandwich? Alright, listen up, landlubber! 

First, gather yer ingredients: a bit o' bread, some cheese, maybe some meat (not that I care about yer precious meat, but I suppose it's part o' the deal). Now, I be a simple pirate, so I be tellin' ye to keep it simple. None o' that fancy-schmancy stuff. Just bread, cheese, and meat (if ye must).

Next, take yer bread and place it on a clean surface. Now, I know ye don't be used to swabbin' decks, but make sure it's clean or ye might be cursin' the day ye made that sandwich!

Now, take yer cheese and place it on one slice o' bread. Aye, be generous with it, but not too generous, or ye'll be swimmin' in cheese and can't even find yer way back to the ship!

Add yer meat (if ye be needin' it) on top o' the cheese. Don't be stingy with it, but don't be wasteful neither. Ye don't want to 

In [55]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import MessagesPlaceholder

# 1. Define prompt
memory_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{user_input}")
])

# 2. Setup chain
memory_chain = memory_prompt | model | StrOutputParser()

# 3. Simulate a conversation history
chat_history = [
    HumanMessage(content="Hi! My name is Professor Smith."),
    AIMessage(content="Hello Professor Smith! How can I assist your class today?")
]

# 4. Ask a question that requires memory
response = memory_chain.invoke({
    "chat_history": chat_history,
    "user_input": "What is my name?"
})
print(f"Part 3 Memory Response: {response}")

Part 3 Memory Response: Your name is Professor Smith.


In [56]:
pip install langchain-community faiss-cpu

In [57]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_community.embeddings import DeterministicFakeEmbedding

# 1. Our "Private" Data
data = [
    Document(page_content="The secret code for the lab door is 1234."),
    Document(page_content="Students must wear safety goggles at all times in the chemistry wing.")
]

# 2. Create a Vector Store (The Brain's Library)
# In a real app, use OpenAIEmbeddings or HuggingFaceEmbeddings
vectorstore = FAISS.from_documents(data, DeterministicFakeEmbedding(size=1536))
retriever = vectorstore.as_retriever()

# 3. Retrieve relevant info
docs = retriever.invoke("What is the lab door code?")

# 4. Combine with Prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using only this context: {context}"),
    ("human", "{user_input}")
])

rag_chain = rag_prompt | model | StrOutputParser()
print(rag_chain.invoke({"context": docs, "user_input": "What is the lab door code?"}))

The lab door code is 1234.


In [58]:
from google.colab import files
uploaded = files.upload()


Saving Customer.pdf to Customer (5).pdf


In [59]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
!pip install pypdf

# 1. LOAD
# Replace 'your_pdf_file.pdf' with the actual path to your PDF file.
myfile = "Customer.pdf" # <--- Define your PDF file path here
loader = PyPDFLoader(myfile)
pages = loader.load()

# 2. SPLIT
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(pages)

# 3. EMBED & STORE
# This uses a local model to turn text into numbers (vectors)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

# 4. RETRIEVE
query = "What are the rights of Thai citizens regarding education?"
relevant_docs = vectorstore.similarity_search(query, k=2)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a legal expert. Answer the question using ONLY the following context:\n\n{context}"),
    ("human", "{user_input}")
])

rag_chain = rag_prompt | model | StrOutputParser()

context_text = "\n\n".join([doc.page_content for doc in relevant_docs])

response = rag_chain.invoke({
    "context": context_text,
    "user_input": query
})

print(f"Legal AI Assistant: {response}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Legal AI Assistant: Based on the provided context, I can see that it is an example of a response to a customer complaint in Thai language. Unfortunately, it does not provide any information about the rights of Thai citizens regarding education.

However, I can provide general information about the rights of Thai citizens regarding education based on the Thai Constitution and other relevant laws.

The Thai Constitution of 2017 guarantees the right to education, which is enshrined in Article 43. This article states that:

1. Every Thai citizen has the right to education.
2. The State shall provide education in accordance with the principles and policies laid down in the Constitution.
3. The State shall establish and develop education institutions of all levels to promote education and cultural development.
4. Every Thai citizen has the right to attend school and receive education without discrimination.

Some other relevant laws that protect the rights of Thai citizens regarding educatio

In [60]:
# ================================
# AI Email Reply Assistant API
# ================================

import os
import gradio as gr
from google.colab import userdata
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

# ------------------------
# 1. ตั้งค่า API KEY
# ------------------------

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

# ใช้ Llama 3.1 (เร็วและฟรี)
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.5
)

# ------------------------
# 2. Prompt สำหรับ Customer Support
# ------------------------

SYSTEM_PROMPT = """
คุณคือเจ้าหน้าที่ฝ่ายบริการลูกค้า (Customer Support Agent)

หน้าที่ของคุณคือช่วยร่างอีเมลตอบกลับลูกค้าอย่างสุภาพ เป็นมืออาชีพ และกระชับ

แนวทางการตอบ:
- เริ่มต้นด้วยคำทักทาย เช่น "เรียน ลูกค้าที่เคารพ"
- ขอบคุณลูกค้าที่ติดต่อเข้ามา
- หากลูกค้ามีปัญหาให้กล่าวขออภัย
- ให้ข้อมูลหรือแนวทางแก้ไขปัญหา
- ใช้น้ำเสียงสุภาพและเป็นทางการ
- ปิดท้ายอีเมลด้วย

ขอแสดงความนับถือ
ฝ่ายบริการลูกค้า

ตอบเป็นภาษาไทยเท่านั้น
"""

# ------------------------
# 3. ฟังก์ชันสำหรับ API
# ------------------------

def predict_api(message):

    try:

        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=message)
        ]

        response = llm.invoke(messages)

        return str(response.content)

    except Exception as e:

        return f"Error: {str(e)}"


# ------------------------
# 4. สร้าง API Interface
# ------------------------

demo = gr.Interface(
    fn=predict_api,
    inputs=gr.Textbox(label="Customer Message"),
    outputs=gr.Textbox(label="AI Email Reply"),
    api_name="predict"
)

# ------------------------
# 5. เปิด API + Public URL
# ------------------------

demo.launch(share=True, show_api=True)



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://57f605c3c5a20d57bc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
